In [ ]:
# Upload file
from google.colab import files
uploaded = files.upload()

Saving calendar.csv.gz to calendar.csv (1).gz
Saving listings.csv.gz to listings.csv (1).gz


In [ ]:
#Import libraries
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import logging
import shutil
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, HTML



In [ ]:
# Linear Regression - Experiment 3
from scipy import stats

# Cleaning price data
df3 = df.copy()
df3['price'] = df3['price'].replace(r'[\$,]', '', regex=True).astype(float)
df3 = df3[df3['price'] > 0]
df3['bathrooms'] = df3['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)

# Adding price/person and review density columns
df3['log_price']       = np.log1p(df3['price'])
df3['price_per_person'] = df3['price'] / df3['accommodates']
df3['review_density']  = df3['number_of_reviews'] / (df3['minimum_nights'] + 1)

# Removing outliers of 3 standard deviations of log_price
before = len(df3)
df3 = df3[np.abs(stats.zscore(df3['log_price'])) < 3]
after = len(df3)
print(f"Rows removed as outliers: {before - after}")

# Adding newly created columns to features
features3 = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'room_type', 'neighbourhood_cleansed',
    'price_per_person',
    'review_density',
]

df3_model = df3[features3 + ['log_price']].dropna()
df3_model = pd.get_dummies(
    df3_model,
    columns=['room_type', 'neighbourhood_cleansed'],
    drop_first=True
)

X3 = df3_model.drop(columns=['log_price'])
y3 = df3_model['log_price']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)


# Train
model3 = LinearRegression()
model3.fit(X3_train, y3_train)
y3_pred = model3.predict(X3_test)

# Metrics
mae3  = mean_absolute_error(y3_test, y3_pred)
rmse3 = np.sqrt(mean_squared_error(y3_test, y3_pred))
r2_3  = r2_score(y3_test, y3_pred)

#Print results
print(mae3)
print(rmse3)
print(r2_3)


Rows removed as outliers: 20
0.1605062510104591
0.21402618206107848
0.8630119551825166
